# Fine-tune Whisper-small on Hindi / Marathi (FLEURS) — zero-shot vs fine-tuned WER
Runs on Kaggle (recommended: GPU T4 x2 or P100, Internet ON) or Colab (T4).
Run cells top to bottom. Change the language in cell 2 only.

In [ ]:
!pip install -q "transformers==4.46.3" "datasets==3.6.0" "accelerate>=0.34" jiwer librosa gradio soundfile

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pathos 0.3.5 requires dill>=0.4.1, but you have dill 0.3.8 which is incompatible.
pathos 0.3.5 requires multiprocess>=0.70.19, but you have multiprocess 0.70.16 which is incompatible.
s3fs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.
litellm 1.93.2 requires tokenizers<1.0,>=0.21.0, but you have tokenizers 0.20.3 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"      
os.environ["WANDB_DISABLED"] = "true"

import re, gc, json, unicodedata
from dataclasses import dataclass
from typing import Any

import numpy as np
import pandas as pd
import torch
import jiwer
from datasets import load_dataset, Audio
from transformers import (
    WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor,
    WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer,
)

LANG_CODE, LANG_NAME, FLEURS_CFG = "hi", "hindi", "hi_in"     
MODEL_ID  = "openai/whisper-small"      
MAX_STEPS = 1000                        
MAX_TRAIN = None                       
MAX_VAL   = 100                         
MAX_TEST  = None                        
HUB_REPO  = None                        

WORKDIR = "/kaggle/working" if os.path.exists("/kaggle/working") else ("/content" if os.path.exists("/content") else ".")
OUT = f"{WORKDIR}/whisper-small-{LANG_CODE}-ft"
assert torch.cuda.is_available(), "Turn on a GPU (Kaggle: Settings > Accelerator; Colab: Runtime > Change runtime type)"
device = "cuda"
print("GPU:", torch.cuda.get_device_name(0), "| output dir:", OUT)

GPU: Tesla T4 | output dir: /kaggle/working/whisper-small-hi-ft


In [ ]:
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_ID)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_ID, language=LANG_NAME, task="transcribe")
processor = WhisperProcessor.from_pretrained(MODEL_ID, language=LANG_NAME, task="transcribe")

def load_split(split):
    ds = load_dataset("google/fleurs", FLEURS_CFG, split=split, trust_remote_code=True)
    ds = ds.select_columns(["audio", "transcription", "num_samples"])
    ds = ds.cast_column("audio", Audio(sampling_rate=16000))
    return ds

train_raw, val_raw, test_raw = load_split("train"), load_split("validation"), load_split("test")
train_raw = train_raw.filter(lambda n: n <= 30 * 16000, input_columns=["num_samples"])
if MAX_TRAIN: train_raw = train_raw.select(range(min(MAX_TRAIN, len(train_raw))))
val_raw = val_raw.select(range(min(MAX_VAL, len(val_raw))))
if MAX_TEST: test_raw = test_raw.select(range(min(MAX_TEST, len(test_raw))))
print({"train": len(train_raw), "val": len(val_raw), "test": len(test_raw)})

def prepare(batch):
    a = batch["audio"]
    batch["input_features"] = feature_extractor(a["array"], sampling_rate=a["sampling_rate"]).input_features[0]
    batch["labels"] = tokenizer(batch["transcription"]).input_ids
    return batch

drop = ["audio", "transcription", "num_samples"]
train_ds = train_raw.map(prepare, remove_columns=drop)
val_ds   = val_raw.map(prepare, remove_columns=drop)
train_ds = train_ds.filter(lambda l: len(l) < 448, input_columns=["labels"])   # max decoder length

preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

... (10 outputs left out)

{'train': 2115, 'val': 100, 'test': 418}


Map:   0%|          | 0/2115 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2115 [00:00<?, ? examples/s]

In [ ]:
def norm(s: str) -> str:
    s = unicodedata.normalize("NFKC", s.lower())
    s = "".join(" " if unicodedata.category(c)[0] in "PS" else c for c in s)   # punctuation + symbols (incl. danda)
    return re.sub(r"\s+", " ", s).strip()

def wer_score(refs, preds) -> float:
    pairs = [(norm(r), norm(p)) for r, p in zip(refs, preds)]
    pairs = [(r, p) for r, p in pairs if r]
    return 100 * jiwer.wer([r for r, _ in pairs], [p for _, p in pairs])

def load_model():
    m = WhisperForConditionalGeneration.from_pretrained(MODEL_ID).to(device)
    m.generation_config.language = LANG_NAME
    m.generation_config.task = "transcribe"
    m.generation_config.forced_decoder_ids = None
    return m

@torch.no_grad()
def transcribe(model, ds, bs=16):
    model.eval()
    preds = []
    for i in range(0, len(ds), bs):
        arrays = [a["array"] for a in ds[i:i + bs]["audio"]]
        feats = feature_extractor(arrays, sampling_rate=16000, return_tensors="pt").input_features.to(device)
        with torch.autocast("cuda", dtype=torch.float16):
            ids = model.generate(feats, language=LANG_NAME, task="transcribe", max_new_tokens=225)
        preds += tokenizer.batch_decode(ids, skip_special_tokens=True)
    return preds

In [ ]:
refs = test_raw["transcription"]
base_model = load_model()
base_preds = transcribe(base_model, test_raw)
base_wer = wer_score(refs, base_preds)
print(f"Zero-shot WER ({MODEL_ID}, {LANG_NAME}): {base_wer:.2f}%")
del base_model; gc.collect(); torch.cuda.empty_cache()

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, 50259], [2, 50359], [3, 50363]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Zero-shot WER (openai/whisper-small, hindi): 66.10%


In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int
    def __call__(self, features):
        inputs = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(inputs, return_tensors="pt")
        labs = [{"input_ids": f["labels"]} for f in features]
        labs = self.processor.tokenizer.pad(labs, return_tensors="pt")
        labels = labs["input_ids"].masked_fill(labs.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

model = load_model()
model.config.use_cache = False                     
collator = DataCollatorSpeechSeq2SeqWithPadding(processor, model.config.decoder_start_token_id)

def compute_metrics(pred):
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str = tokenizer.batch_decode(pred.predictions, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"wer": wer_score(label_str, pred_str)}

args = Seq2SeqTrainingArguments(
    output_dir=OUT,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,                 
    learning_rate=1e-5,
    warmup_steps=50,
    max_steps=MAX_STEPS,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=250,
    eval_steps=250,
    logging_steps=25,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to="none",
)
trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=collator, compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)
trainer.train()

max_steps is given, it will override any value given in num_train_epochs


Step,Training Loss,Validation Loss,Wer
250,0.191100,0.282424,29.079861
500,0.065700,0.288876,27.300347
750,0.020000,0.321993,26.996528
1000,0.009000,0.346509,27.300347


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

TrainOutput(global_step=1000, training_loss=0.12150189182162285, metrics={'train_runtime': 4654.8948, 'train_samples_per_second': 3.437, 'train_steps_per_second': 0.215, 'total_flos': 4.6072659161088e+18, 'train_loss': 0.12150189182162285, 'epoch': 7.547169811320755})

In [ ]:
trainer.save_model(OUT)
processor.save_pretrained(OUT)

ft_model = trainer.model
ft_model.config.use_cache = True
ft_preds = transcribe(ft_model, test_raw)
ft_wer = wer_score(refs, ft_preds)

print("\n================ RESULTS ================")
print(f"Model: {MODEL_ID} | Language: {LANG_NAME} | Test utterances: {len(refs)} | Train steps: {MAX_STEPS}")
print(f"Zero-shot WER : {base_wer:.2f}%")
print(f"Fine-tuned WER: {ft_wer:.2f}%")
print(f"Absolute reduction: {base_wer - ft_wer:.2f} points ({100*(base_wer-ft_wer)/base_wer:.1f}% relative)")

def safe_wer(r, p):
    r, p = norm(r), norm(p)
    return jiwer.wer(r, p) if r else np.nan

df = pd.DataFrame({"reference": refs, "zero_shot": base_preds, "fine_tuned": ft_preds})
df["wer_zero_shot"] = [safe_wer(r, p) for r, p in zip(df.reference, df.zero_shot)]
df["wer_fine_tuned"] = [safe_wer(r, p) for r, p in zip(df.reference, df.fine_tuned)]
df.to_csv(f"{WORKDIR}/error_analysis.csv", index=False)
json.dump({"model": MODEL_ID, "language": LANG_NAME, "dataset": f"google/fleurs {FLEURS_CFG} test",
           "n_test": len(refs), "train_steps": MAX_STEPS,
           "wer_zero_shot": round(base_wer, 2), "wer_fine_tuned": round(ft_wer, 2)},
          open(f"{WORKDIR}/results.json", "w"), indent=2)
print(f"\nSaved {WORKDIR}/error_analysis.csv and results.json")
print("\nWorst 10 fine-tuned utterances (for error analysis):")
print(df.sort_values("wer_fine_tuned", ascending=False).head(10)[["reference", "fine_tuned", "wer_fine_tuned"]].to_string())


================ RESULTS ================
Model: openai/whisper-small | Language: hindi | Test utterances: 418 | Train steps: 1000
Zero-shot WER : 66.10%
Fine-tuned WER: 26.56%
Absolute reduction: 39.54 points (59.8% relative)

Saved /kaggle/working/error_analysis.csv and results.json

Worst 10 fine-tuned utterances (for error analysis):
                                                                                                                                                                                      reference                                                                                                                                                              fine_tuned  wer_fine_tuned
316                                                                     तथा क्रांति के बाद व्यवसाय सभी पुरुष आवेदकों के लिए खुले थे जिसमें सबसे महत्त्वाकांक्षी और सफल होने वाले सफल हो सकते थे                                               तुफा प्राल्त के बैर्मादी हो शायसा द्वी पूरु आ

In [ ]:
def get_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return None

if HUB_REPO:
    tok = get_hf_token()
    assert tok, "Add a write-access HF_TOKEN secret first"
    ft_model.push_to_hub(HUB_REPO, token=tok)
    processor.push_to_hub(HUB_REPO, token=tok)
    print("Pushed to https://huggingface.co/" + HUB_REPO)

In [9]:
# 9. Gradio demo (mic or upload) — a public link appears for ~72 h
import gradio as gr, librosa
from transformers import pipeline

asr = pipeline("automatic-speech-recognition", model=OUT, device=0, chunk_length_s=30,
               generate_kwargs={"language": LANG_NAME, "task": "transcribe"})

def run(path):
    if path is None:
        return ""
    audio, _ = librosa.load(path, sr=16000)
    return asr({"raw": audio, "sampling_rate": 16000})["text"]

gr.Interface(
    fn=run,
    inputs=gr.Audio(sources=["microphone", "upload"], type="filepath"),
    outputs=gr.Textbox(label="Transcript"),
    title=f"Whisper-small fine-tuned for {LANG_NAME.title()} (FLEURS)",
    description=f"Zero-shot WER {base_wer:.1f}% -> fine-tuned WER {ft_wer:.1f}% on FLEURS {FLEURS_CFG} test.",
).launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://7467e543e42baaa21d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
